In [3]:
from transformers import AutoModel,AutoTokenizer
import torch
import torch.nn.functional as F
model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModel.from_pretrained(model_name)
documents=[
    "減脂晚餐建議吃高蛋白、低油脂、適量碳水",
    "胸肌訓練可以安排槓鈴握推、上斜啞鈴握推、夾胸",
    "深蹲可以訓練股四頭肌、臀部與核心穩定",
    "增肌期需要足夠蛋白質、熱量盈餘及重量練"
]
def encode_texts(texts):
    inputs=tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )
    with torch.no_grad():
        outputs=model(**inputs)
    token_embeddings=outputs.last_hidden_state
    attention_mask=inputs["attention_mask"]

    mask=attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    masked_embeddings=token_embeddings*mask

    sum_embeddings=torch.sum(masked_embeddings,dim=1)
    sum_mask=torch.clamp(mask.sum(dim=1),min=1e-9)

    sentence_embeddings=sum_embeddings/sum_mask
    sentence_embeddings=F.normalize(sentence_embeddings,p=2,dim=1)

    return sentence_embeddings
doc_embeddings=encode_texts(documents)
def semantic_search(query,documents,doc_embeddings,top_k=2):
    query_embeddings=encode_texts([query])
    scores=torch.matmul(query_embeddings,doc_embeddings.T)
    topk_values,topk_indices=torch.topk(scores,k=top_k,dim=1)
    result=[]
    for i in range(top_k):
        idx=topk_indices[0][i].item()
        score=topk_values[0][i].item()
        result.append({
            "documents":documents[idx],
            "score":score
        })
    return result
query="我今天想練胸,應該做哪些動作?"
results=semantic_search(query,documents,doc_embeddings,top_k=2)
print("query",query)
print(f"\nTop results:")
for i,result in enumerate(results):
    print(f"Rank{i+1}")
    print("documents",result["documents"])
    print("score",round(result["score"],4))
    print()



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4909.60it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


query 我今天想練胸,應該做哪些動作?

Top results:
Rank1
documents 胸肌訓練可以安排槓鈴握推、上斜啞鈴握推、夾胸
score 0.5494

Rank2
documents 深蹲可以訓練股四頭肌、臀部與核心穩定
score 0.4254

